# 04 - IndoBERT (Google Colab dengan GPU)

Notebook ini melatih dan menguji **IndoBERT** untuk membedakan komentar **normal (0)** dan **promosi judi (1)**.
Berbeda dari TF-IDF yang hanya menghitung bobot kata, IndoBERT membaca kata beserta konteks kalimatnya.

**Alur:** dataset clean -> split -> tokenisasi -> fine-tuning IndoBERT -> evaluasi -> analisis kesalahan -> simpan

Notebook ini **terpisah dari notebook 01 - 03**. File hasilnya memakai awalan `indobert_` dan tidak menimpa model atau hasil
notebook lain.

| | |
|---|---|
| **Input wajib** | `data/dataset_clean.csv` (hasil notebook 01) |
| **Input opsional** | `outputs/metrik_evaluasi.csv` (notebook 02), `outputs/hasil_prediksi.csv` (notebook 03) |
| **Output** | `models_indobert/`, `outputs/indobert_*.csv`, `outputs/indobert_*.png` |
| **Untuk laporan** | Tabel 4.1 - 4.9 dan Gambar 4.1 - 4.4; tabel dan gambar dari langkah yang dilewati tidak akan muncul |

## Cara menjalankan di Colab

1. Unggah folder `pipeline_judol` ke **Google Drive**, tepat di `MyDrive/pipeline_judol`.
2. Buka notebook ini dari Drive dengan Google Colab.
3. Pilih `Runtime -> Change runtime type -> T4 GPU`.
4. Untuk percobaan pertama, ubah `UJI_CEPAT = True`, lalu `Run All`. Izinkan akses Drive jika diminta.
5. Jika tidak ada error, kembalikan `UJI_CEPAT = False`, lalu `Run All` lagi.

## 1. Pengaturan

Semua pilihan ada di sel berikut. Nilai `False` berarti langkah tersebut **dilewati (bypass)**.

| Pengaturan | Jika `True` | Jika `False` (bypass) |
|---|---|---|
| `UJI_CEPAT` | memakai sebagian kecil data dan 1 epoch untuk mengecek alur | memakai seluruh data |
| `LATIH_MODEL` | melatih IndoBERT | melewati training dan memuat model tersimpan |
| `EVALUASI_DATA_UJI` | menghitung metrik, grafik, dan analisis kesalahan | melewati evaluasi |
| `PREDIKSI_HASIL_SCRAPING` | menambahkan prediksi IndoBERT ke hasil notebook 03 | melewati prediksi |
| `SIMPAN_MODEL` | menyimpan model (sekitar 400 MB) | model hilang saat sesi berakhir |
| `PAKAI_DRIVE` | membaca dan menyimpan file di Google Drive | memakai folder sementara Colab; dataset diunggah manual dan `outputs/` diunduh sebagai ZIP |
| `UNDERSAMPLING` | menyeimbangkan data latih seperti notebook 02 | memakai data latih apa adanya |

**Pilihan teks input (`KOLOM_TEKS`):**
- `"komentar_clean"`: teks yang sama dengan notebook 02, sehingga perbandingannya setara.
- `"komentar"`: teks asli tanpa cleaning. Pilihan ini melewati cleaning dan mempertahankan stopword yang bisa membantu BERT memahami konteks.

Bentuk teksnya berbeda, tetapi baris data ujinya tetap sama.

In [ ]:
# Jalur proses: True = jalankan, False = lewati
UJI_CEPAT = False
LATIH_MODEL = True
EVALUASI_DATA_UJI = True
PREDIKSI_HASIL_SCRAPING = True
SIMPAN_MODEL = True
PAKAI_DRIVE = True
UNDERSAMPLING = True

# Data dan model
KOLOM_TEKS = "komentar_clean"      # "komentar_clean" atau "komentar"
FOLDER_DRIVE = "/content/drive/MyDrive/pipeline_judol"
NAMA_MODEL_DASAR = "indobenchmark/indobert-base-p1"

# Parameter training
MAX_LENGTH = 64                    # panjang maksimum token per komentar
BATCH_SIZE = 16                    # turunkan menjadi 8 jika muncul error "CUDA out of memory"
EPOCHS = 3
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
RANDOM_STATE = 42

assert KOLOM_TEKS in ("komentar_clean", "komentar"), "Pilihan KOLOM_TEKS tidak dikenal"

## 2. Setup

In [ ]:
import importlib.util

# Colab biasanya sudah memiliki library ini; yang belum ada saja yang dipasang
for paket in ["torch", "transformers", "sklearn", "pandas", "matplotlib", "jinja2", "tqdm"]:
    if importlib.util.find_spec(paket) is None:
        nama_pip = "scikit-learn" if paket == "sklearn" else paket
        %pip install -q {nama_pip}

In [ ]:
import json
import os
import random
import shutil
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, classification_report, confusion_matrix, f1_score,
                             precision_score, recall_score, roc_auc_score, roc_curve)
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer

DI_COLAB = "google.colab" in sys.modules

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
torch.cuda.manual_seed_all(RANDOM_STATE)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("PERHATIAN: GPU tidak terdeteksi, training di CPU akan sangat lama.")
    print("Di Colab, pilih Runtime -> Change runtime type -> T4 GPU.")

pd.set_option("display.max_colwidth", 90)
plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 200,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.titleweight": "bold",
})
WARNA_UTAMA, WARNA_KEDUA = "#2a78d6", "#eb6834"
NAMA_LABEL = {0: "Normal (0)", 1: "Promosi judi (1)"}


def tampil(tabel, judul, file=None, sembunyikan_index=False):
    """Menampilkan tabel yang rapi untuk di-screenshot. Jika file diisi, tabel juga disimpan ke outputs/."""
    if file:
        tabel.to_csv(f"outputs/{file}", index=not sembunyikan_index)
    gaya = (tabel.style.set_caption(judul).format(precision=4, na_rep="-")
            .set_properties(**{"text-align": "left", "white-space": "normal"})
            .set_table_styles([
                {"selector": "caption", "props": "caption-side: top; font-weight: bold; font-size: 1.05em; "
                                                 "text-align: left; padding-bottom: 6px; color: inherit;"},
                {"selector": "th", "props": "text-align: left;"},
            ]))
    display(gaya.hide(axis="index") if sembunyikan_index else gaya)

## 3. Menyiapkan folder kerja

Jika `PAKAI_DRIVE = True`, Colab meminta izin mengakses Google Drive agar hasil tidak hilang saat sesi berakhir.

In [ ]:
if DI_COLAB and PAKAI_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(FOLDER_DRIVE, exist_ok=True)
    os.chdir(FOLDER_DRIVE)

os.makedirs("data", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

# Hasil uji cepat dipisahkan agar tidak menimpa hasil sebenarnya
AWALAN = "indobert_uji_cepat_" if UJI_CEPAT else "indobert_"
FOLDER_MODEL = "models_indobert_uji_cepat" if UJI_CEPAT else "models_indobert"
if UJI_CEPAT:
    EPOCHS = 1

# Saat training dilewati, gunakan pengaturan yang tersimpan bersama model
FILE_KONFIGURASI = os.path.join(FOLDER_MODEL, "konfigurasi_indobert.json")
if not LATIH_MODEL:
    if not os.path.exists(FILE_KONFIGURASI):
        raise FileNotFoundError(f"Model tersimpan tidak ditemukan di {FOLDER_MODEL}/. Jalankan dulu dengan LATIH_MODEL = True dan SIMPAN_MODEL = True.")
    with open(FILE_KONFIGURASI) as f:
        tersimpan = json.load(f)
    KOLOM_TEKS, MAX_LENGTH = tersimpan["kolom_teks"], tersimpan["max_length"]
    print("Training dilewati. Memuat model dari", FOLDER_MODEL)

status = lambda aktif: "dijalankan" if aktif else "dilewati"
tampil(pd.DataFrame({
    "Pengaturan": ["Model dasar", "Kolom teks", "Panjang maksimum", "Batch size", "Epoch", "Learning rate", "Undersampling",
                   "Training", "Evaluasi data uji", "Prediksi hasil scraping", "Mode uji cepat", "Folder kerja"],
    "Nilai": [NAMA_MODEL_DASAR, KOLOM_TEKS, MAX_LENGTH, BATCH_SIZE, EPOCHS, LEARNING_RATE, status(UNDERSAMPLING),
              status(LATIH_MODEL), status(EVALUASI_DATA_UJI), status(PREDIKSI_HASIL_SCRAPING), status(UJI_CEPAT), os.getcwd()],
}).astype(str), "Tabel 4.1. Pengaturan percobaan IndoBERT", sembunyikan_index=True)

## 4. Memuat dan membagi data

Pembagian **data uji** memakai cara dan `random_state` yang sama dengan notebook 02, sehingga baris uji sama jika notebook 02
dijalankan dengan pengaturan bawaan.

Sebanyak 10% data latih dipisahkan menjadi **data validasi** untuk memilih epoch terbaik. Data uji tetap tidak dipakai selama
training. Karena itu, data latih IndoBERT sedikit lebih sedikit daripada notebook 02.

In [ ]:
PERLU_DATA = LATIH_MODEL or EVALUASI_DATA_UJI

if PERLU_DATA:
    FILE_DATA = "data/dataset_clean.csv"
    if not os.path.exists(FILE_DATA):
        if not DI_COLAB:
            raise FileNotFoundError(f"{FILE_DATA} tidak ditemukan. Jalankan notebook 01 terlebih dahulu.")
        from google.colab import files
        print("Pilih file dataset_clean.csv dari komputer Anda.")
        unggahan = files.upload()
        if "dataset_clean.csv" not in unggahan:
            raise FileNotFoundError("Nama file yang diunggah harus dataset_clean.csv")
        shutil.move("dataset_clean.csv", FILE_DATA)

    df = pd.read_csv(FILE_DATA)
    df[KOLOM_TEKS] = df[KOLOM_TEKS].fillna("").astype(str)

    latih_penuh, uji = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=RANDOM_STATE)
    latih, validasi = train_test_split(latih_penuh, test_size=0.1, stratify=latih_penuh["label"], random_state=RANDOM_STATE)
    jumlah_sebelum = latih["label"].value_counts()

    if UNDERSAMPLING:
        n_min = latih["label"].value_counts().min()
        latih = latih.groupby("label").sample(n=n_min, random_state=RANDOM_STATE)
    latih = latih.sample(frac=1, random_state=RANDOM_STATE)

    if UJI_CEPAT:
        latih = latih.sample(min(200, len(latih)), random_state=RANDOM_STATE)
        validasi = validasi.sample(min(100, len(validasi)), random_state=RANDOM_STATE)
        uji = uji.sample(min(200, len(uji)), random_state=RANDOM_STATE)
        print("UJI CEPAT: hanya sebagian kecil data dipakai. Nilai evaluasi tidak boleh dipakai di laporan.")

    pembagian = pd.DataFrame({"Latih (sebelum penyeimbangan)": jumlah_sebelum, "Latih (dipakai)": latih["label"].value_counts(),
                              "Validasi": validasi["label"].value_counts(), "Uji": uji["label"].value_counts()}).rename(index=NAMA_LABEL)
    pembagian.loc["Total"] = pembagian.sum()
    tampil(pembagian.rename_axis("Label"), "Tabel 4.2. Pembagian data", f"{AWALAN}pembagian_data.csv")
else:
    print("Data training tidak dimuat karena training dan evaluasi dilewati.")

## 5. Memuat tokenizer dan model

- **Tokenizer** memecah teks menjadi token dan mengubahnya menjadi angka sesuai kosakata IndoBERT.
- **Model** IndoBERT diberi lapisan klasifikasi baru berisi 2 kelas. Peringatan bahwa sebagian bobot baru dibuat adalah hal
  yang wajar, karena lapisan ini memang baru dan akan dilatih.

In [ ]:
PERLU_MODEL = LATIH_MODEL or EVALUASI_DATA_UJI or PREDIKSI_HASIL_SCRAPING

if PERLU_MODEL:
    sumber_model = NAMA_MODEL_DASAR if LATIH_MODEL else FOLDER_MODEL
    tokenizer = AutoTokenizer.from_pretrained(sumber_model)
    model = AutoModelForSequenceClassification.from_pretrained(sumber_model, num_labels=2).to(DEVICE)
    print(f"Model dimuat dari: {sumber_model}")
    print(f"Jumlah parameter: {sum(p.numel() for p in model.parameters()):,}")


def buat_loader(daftar_teks, label=None, acak=False):
    token = tokenizer(list(daftar_teks), padding="max_length", truncation=True, max_length=MAX_LENGTH, return_tensors="pt")
    tensor = [token["input_ids"], token["attention_mask"]]
    if label is not None:
        tensor.append(torch.tensor(list(label), dtype=torch.long))
    return DataLoader(TensorDataset(*tensor), batch_size=BATCH_SIZE, shuffle=acak)


def prediksi_peluang(daftar_teks):
    """Mengembalikan peluang kelas promosi judi untuk setiap teks."""
    model.eval()
    hasil = []
    with torch.no_grad():
        for input_ids, attention_mask in buat_loader(daftar_teks):
            logits = model(input_ids=input_ids.to(DEVICE), attention_mask=attention_mask.to(DEVICE)).logits
            hasil.append(torch.softmax(logits, dim=1)[:, 1].cpu().numpy())
    return np.concatenate(hasil) if hasil else np.array([])


if PERLU_MODEL and PERLU_DATA:
    # Contoh tokenisasi agar proses ini terlihat
    contoh_teks = (latih if LATIH_MODEL else uji)[KOLOM_TEKS].iloc[0]
    print("\nContoh teks :", contoh_teks)
    print("Hasil token :", tokenizer.tokenize(contoh_teks))

## 6. Training (fine-tuning)

Setiap **epoch**, model membaca seluruh data latih per kelompok kecil (*batch*), menghitung kesalahannya (*loss*), lalu
memperbaiki bobotnya. Setelah setiap epoch, model dinilai dengan data validasi. Bobot dengan **F1 validasi tertinggi** yang
dipakai pada tahap berikutnya.

**Gambar 4.1. Kurva training**

In [ ]:
riwayat = []

if LATIH_MODEL:
    loader_latih = buat_loader(latih[KOLOM_TEKS], latih["label"], acak=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

    # Learning rate naik pada 10% langkah pertama, lalu turun perlahan hingga 0
    total_langkah = len(loader_latih) * EPOCHS
    langkah_pemanasan = int(0.1 * total_langkah)

    def faktor_lr(langkah):
        if langkah < langkah_pemanasan:
            return (langkah + 1) / max(1, langkah_pemanasan)
        return max(0.0, (total_langkah - langkah) / max(1, total_langkah - langkah_pemanasan))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, faktor_lr)

    f1_terbaik, bobot_terbaik, epoch_terbaik = -1.0, None, 0
    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss = 0.0
        for input_ids, attention_mask, label in tqdm(loader_latih, desc=f"Epoch {epoch}/{EPOCHS}"):
            optimizer.zero_grad()
            keluaran = model(input_ids=input_ids.to(DEVICE), attention_mask=attention_mask.to(DEVICE), labels=label.to(DEVICE))
            keluaran.loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            total_loss += keluaran.loss.item()

        peluang_validasi = prediksi_peluang(validasi[KOLOM_TEKS])
        f1_validasi = f1_score(validasi["label"], (peluang_validasi >= 0.5).astype(int), zero_division=0)
        riwayat.append({"epoch": epoch, "loss_latih": total_loss / len(loader_latih), "f1_validasi": f1_validasi})
        print(f"Epoch {epoch}: loss latih {riwayat[-1]['loss_latih']:.4f}, F1 validasi {f1_validasi:.4f}")

        if f1_validasi > f1_terbaik:
            f1_terbaik, epoch_terbaik = f1_validasi, epoch
            bobot_terbaik = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(bobot_terbaik)
    print(f"\nBobot terbaik berasal dari epoch {epoch_terbaik} (F1 validasi {f1_terbaik:.4f})")

    riwayat_df = pd.DataFrame(riwayat)
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
    for ax, kolom, judul in [(axes[0], "loss_latih", "Loss data latih"), (axes[1], "f1_validasi", "F1 data validasi")]:
        ax.plot(riwayat_df["epoch"], riwayat_df[kolom], marker="o", markersize=8, linewidth=2, color=WARNA_UTAMA)
        ax.set_xticks(riwayat_df["epoch"])
        ax.set(title=judul, xlabel="Epoch", ylabel=judul.split()[0])
    plt.tight_layout()
    plt.savefig(f"outputs/{AWALAN}kurva_training.png", bbox_inches="tight")
    plt.show()

    tampil(riwayat_df, "Tabel 4.3. Riwayat training per epoch", f"{AWALAN}riwayat_training.csv", sembunyikan_index=True)
else:
    print("Training dilewati (LATIH_MODEL = False)")

## 7. Menyimpan model

Model disimpan bersama tokenizer dan pengaturannya, sehingga dapat dimuat kembali dengan `LATIH_MODEL = False` tanpa training ulang.

In [ ]:
if LATIH_MODEL and SIMPAN_MODEL:
    os.makedirs(FOLDER_MODEL, exist_ok=True)
    model.save_pretrained(FOLDER_MODEL)
    tokenizer.save_pretrained(FOLDER_MODEL)
    with open(FILE_KONFIGURASI, "w") as f:
        json.dump({"model_dasar": NAMA_MODEL_DASAR, "kolom_teks": KOLOM_TEKS, "max_length": MAX_LENGTH,
                   "epoch_terbaik": epoch_terbaik, "f1_validasi": f1_terbaik, "undersampling": UNDERSAMPLING}, f, indent=2)
    print(f"Model tersimpan di {FOLDER_MODEL}/")
else:
    print("Penyimpanan model dilewati")

## 8. Evaluasi pada data uji

Precision, recall, dan F1 dihitung untuk kelas promosi judi. Metriknya sama dengan notebook 02: `tp` = promosi tertangkap,
`tn` = normal benar, `fp` = normal yang salah ditandai promosi, dan `fn` = promosi yang lolos.

In [ ]:
if EVALUASI_DATA_UJI:
    y_uji = uji["label"].to_numpy()
    skor_uji = prediksi_peluang(uji[KOLOM_TEKS])
    pred_uji = (skor_uji >= 0.5).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_uji, pred_uji, labels=[0, 1]).ravel()
    metrik = pd.DataFrame([{
        "accuracy": accuracy_score(y_uji, pred_uji), "precision": precision_score(y_uji, pred_uji, zero_division=0),
        "recall": recall_score(y_uji, pred_uji, zero_division=0), "f1": f1_score(y_uji, pred_uji, zero_division=0),
        "roc_auc": roc_auc_score(y_uji, skor_uji) if len(set(y_uji)) == 2 else np.nan,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }], index=pd.Index(["indobert"], name="model"))
    tampil(metrik, "Tabel 4.4. Hasil evaluasi IndoBERT pada data uji", f"{AWALAN}metrik.csv")

    laporan = pd.DataFrame(classification_report(y_uji, pred_uji, labels=[0, 1], target_names=["Normal", "Promosi judi"],
                                                 output_dict=True, zero_division=0)).T
    laporan["support"] = laporan["support"].astype(int)
    if "accuracy" in laporan.index:
        laporan.loc["accuracy", ["precision", "recall"]] = np.nan
        laporan.loc["accuracy", "support"] = len(y_uji)
    tampil(laporan.rename_axis("Kelas"), "Tabel 4.5. Laporan klasifikasi IndoBERT")
else:
    print("Evaluasi dilewati (EVALUASI_DATA_UJI = False)")

**Gambar 4.2. Confusion matrix dan Gambar 4.3. Kurva ROC**

In [ ]:
if EVALUASI_DATA_UJI:
    fig, ax = plt.subplots(figsize=(4.2, 3.6))
    ConfusionMatrixDisplay(confusion_matrix(y_uji, pred_uji, labels=[0, 1]), display_labels=["Normal", "Promosi"]).plot(
        ax=ax, cmap="Blues", colorbar=False, values_format="d")
    ax.grid(False)
    ax.set(title="Confusion matrix IndoBERT", xlabel="Prediksi", ylabel="Sebenarnya")
    plt.tight_layout()
    plt.savefig(f"outputs/{AWALAN}confusion_matrix.png", bbox_inches="tight")
    plt.show()

    if len(set(y_uji)) == 2:
        fpr, tpr, _ = roc_curve(y_uji, skor_uji)
        fig, ax = plt.subplots(figsize=(5, 4.2))
        ax.plot(fpr, tpr, linewidth=2, color=WARNA_UTAMA, label=f"indobert (AUC {metrik.loc['indobert', 'roc_auc']:.3f})")
        ax.plot([0, 1], [0, 1], linestyle="--", linewidth=1, color="#52514e", label="tebakan acak")
        ax.set(title="Kurva ROC IndoBERT", xlabel="False positive rate", ylabel="True positive rate")
        ax.legend(frameon=False, loc="lower right")
        plt.tight_layout()
        plt.savefig(f"outputs/{AWALAN}roc_curve.png", bbox_inches="tight")
        plt.show()

## 9. Analisis kesalahan

Komentar diurutkan dari yang **paling yakin tetapi salah**. `peluang_promosi` adalah keyakinan model bahwa komentar tersebut
promosi judi (0 - 1). Periksa juga labelnya, karena sebagian kesalahan bisa berasal dari label dataset yang keliru.

In [ ]:
if EVALUASI_DATA_UJI:
    hasil_uji = uji[["komentar", "komentar_clean", "label"]].copy()
    hasil_uji["prediksi"] = pred_uji
    hasil_uji["peluang_promosi"] = skor_uji

    salah_fp = hasil_uji[(hasil_uji["label"] == 0) & (hasil_uji["prediksi"] == 1)].sort_values("peluang_promosi", ascending=False)
    salah_fn = hasil_uji[(hasil_uji["label"] == 1) & (hasil_uji["prediksi"] == 0)].sort_values("peluang_promosi")
    kolom = ["komentar", "komentar_clean", "peluang_promosi"]

    tampil(salah_fp[kolom].head(10), "Tabel 4.6. Contoh false positive: komentar normal yang ditandai promosi", sembunyikan_index=True)
    tampil(salah_fn[kolom].head(10), "Tabel 4.7. Contoh false negative: komentar promosi yang lolos", sembunyikan_index=True)
    pd.concat([salah_fp, salah_fn]).to_csv(f"outputs/{AWALAN}kesalahan.csv", index=False)

## 10. Perbandingan dengan model notebook 02

Bagian ini dilewati otomatis jika `outputs/metrik_evaluasi.csv` tidak ada. Perbandingan hanya setara jika notebook 02 memakai
pengaturan bawaan, `KOLOM_TEKS` memakai `"komentar_clean"`, dan `UJI_CEPAT = False`.

**Gambar 4.4. Perbandingan F1 antar model**

In [ ]:
FILE_METRIK_NB2 = "outputs/metrik_evaluasi.csv"

if EVALUASI_DATA_UJI and not UJI_CEPAT and os.path.exists(FILE_METRIK_NB2):
    kolom_metrik = ["accuracy", "precision", "recall", "f1", "roc_auc", "fp", "fn"]
    banding = pd.concat([pd.read_csv(FILE_METRIK_NB2, index_col="model")[kolom_metrik], metrik[kolom_metrik]])
    banding = banding.sort_values("f1", ascending=False)

    urut = banding.sort_values("f1")
    fig, ax = plt.subplots(figsize=(8, 3.2))
    warna = [WARNA_KEDUA if n == "indobert" else WARNA_UTAMA for n in urut.index]
    ax.barh(urut.index, urut["f1"], color=warna, height=0.55)
    for i, v in enumerate(urut["f1"]):
        ax.text(v, i, f"  {v:.3f}", va="center")
    ax.set_xlim(0, 1.1)
    ax.set(title="F1 pada data uji", xlabel="F1")
    ax.grid(axis="y", visible=False)
    plt.tight_layout()
    plt.savefig(f"outputs/{AWALAN}perbandingan_f1.png", bbox_inches="tight")
    plt.show()

    tampil(banding, "Tabel 4.8. Perbandingan IndoBERT dengan model notebook 02", f"{AWALAN}perbandingan.csv")
else:
    print("Perbandingan dilewati")

## 11. Prediksi komentar hasil scraping

Bagian ini membaca `outputs/hasil_prediksi.csv` dari notebook 03 dan menambahkan kolom `predicted_label_indobert`. Hasilnya
disimpan sebagai **file baru**, sehingga file notebook 03 tidak berubah. Bagian ini dilewati otomatis jika file tersebut tidak ada.

In [ ]:
FILE_SCRAPING = "outputs/hasil_prediksi.csv"

if PREDIKSI_HASIL_SCRAPING and os.path.exists(FILE_SCRAPING):
    hasil_scraping = pd.read_csv(FILE_SCRAPING)
    teks_scraping = hasil_scraping[KOLOM_TEKS].fillna("").astype(str)
    hasil_scraping["predicted_label_indobert"] = (prediksi_peluang(teks_scraping) >= 0.5).astype(int)
    hasil_scraping.to_csv(f"outputs/{AWALAN}hasil_prediksi.csv", index=False)

    n_promosi = int(hasil_scraping["predicted_label_indobert"].sum())
    print(f"IndoBERT menandai {n_promosi:,} dari {len(hasil_scraping):,} komentar sebagai promosi. Angka ini bukan ukuran akurasi.")
    ditandai = hasil_scraping.loc[hasil_scraping["predicted_label_indobert"] == 1, ["video_id", "komentar"]]
    tampil(ditandai.head(10), "Tabel 4.9. Contoh komentar yang ditandai IndoBERT (perlu diperiksa manual)", sembunyikan_index=True)
else:
    print("Prediksi hasil scraping dilewati")

## 12. Ringkasan untuk laporan

Kalimat berikut dibuat otomatis, dan dapat disalin lalu disesuaikan.

In [ ]:
if UJI_CEPAT:
    print("Mode UJI_CEPAT aktif: alur sudah dicek, tetapi nilainya tidak boleh dipakai di laporan.")
    print("Ubah UJI_CEPAT = False, lalu jalankan ulang.")
elif EVALUASI_DATA_UJI:
    m = metrik.loc["indobert"]
    print(f"Model {NAMA_MODEL_DASAR} dilatih dengan teks kolom {KOLOM_TEKS}.")
    if LATIH_MODEL:
        print(f"Data latih berisi {len(latih):,} komentar, data validasi {len(validasi):,} komentar, dan bobot terbaik berasal dari epoch {epoch_terbaik}.")
    print(f"Pada {len(uji):,} komentar data uji, IndoBERT memperoleh F1 {m['f1']:.3f}, precision {m['precision']:.3f}, recall {m['recall']:.3f}, dan accuracy {m['accuracy']:.3f}.")
    print(f"Model ini menghasilkan {int(m['fp'])} false positive dan {int(m['fn'])} false negative.")
else:
    print("Ringkasan dilewati karena evaluasi tidak dijalankan.")

print("\nFile IndoBERT di outputs/:")
for f in sorted(os.listdir("outputs")):
    if f.startswith(AWALAN):
        print(" -", f)

# Tanpa Drive, file Colab hilang saat sesi berakhir: unduh outputs/ sebagai ZIP
if DI_COLAB and not PAKAI_DRIVE:
    from google.colab import files
    files.download(shutil.make_archive("hasil_indobert", "zip", "outputs"))